# TinyGPT v3: RoPE, GQA, Mixed-Precision & Gradient Checkpointing

This notebook walks through the four modernizations introduced in TinyGPT v3
(`tiny_gpt_v3.py`), with live demonstrations of each feature.

**Prerequisites:** `numpy` only — no PyTorch, no TensorFlow (ADR-001).

**Sections:**
1. RoPE — Rotary Position Embeddings
2. GQA — Grouped-Query Attention
3. Mixed-Precision (float16 forward)
4. Gradient Checkpointing
5. 4M-parameter config & forward pass
6. Gradient check (finite-difference verification)
7. Spectral analysis (RMT diagnostics)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from tiny_gpt_v3 import (
    TinyGPTV3, TinyGPTV3Config, config_4m, config_small,
    rope_apply, rope_backward, rope_freqs,
    gqa_forward, gqa_backward,
)
print('NumPy:', np.__version__)

## 1. RoPE — Rotary Position Embeddings

RoPE applies a rotation matrix to each (even, odd) pair of the Q and K
vectors. The rotation angle depends on the token position, so the dot
product `Q_i · K_j` encodes the **relative** distance `i - j`.

Key property: RoPE is exactly unitary, so it preserves the L2 norm of
each pair.

In [ ]:
# Visualize the rotation angles for head_dim=32.
freqs = rope_freqs(32, base=10000.0)
print(f'Inverse frequencies (first 8): {freqs[:8]}')
print(f'Shape: {freqs.shape} (head_dim/2 = {32//2})')

# Verify norm preservation.
x = np.random.randn(8, 4, 32).astype(np.float64)
out, (cos, sin) = rope_apply(x)
x_norm = np.sqrt(x[..., 0::2]**2 + x[..., 1::2]**2)
out_norm = np.sqrt(out[..., 0::2]**2 + out[..., 1::2]**2)
print(f'\nNorm preservation: max error = {np.max(np.abs(x_norm - out_norm)):.2e}')

# Verify identity at position 0.
x0 = np.random.randn(1, 4, 32)
out0, _ = rope_apply(x0, seq_offset=0)
print(f'Identity at pos 0: max error = {np.max(np.abs(x0 - out0)):.2e}')

## 2. GQA — Grouped-Query Attention

With GQA, `n_kv_heads < n_heads`. Each KV head is shared by
`n_heads / n_kv_heads` query heads. This reduces KV projection
parameters and the KV cache at inference time.

For the 4M config: `n_heads=6, n_kv_heads=2` → group size 3.

In [ ]:
T, nh, nkv, hd = 8, 6, 2, 32
q = np.random.randn(T, nh, hd)
k = np.random.randn(T, nkv, hd)
v = np.random.randn(T, nkv, hd)

out, cache = gqa_forward(q, k, v, nh, nkv, hd, rope=False)
print(f'Q shape: {q.shape} (T, n_heads, head_dim)')
print(f'K shape: {k.shape} (T, n_kv_heads, head_dim)')
print(f'Output shape: {out.shape} (T, n_heads, head_dim)')
print(f'Group size: {nh // nkv} (each KV head serves {nh//nkv} query heads)')

# Backward pass.
dout = np.random.randn(*out.shape)
dq, dk, dv = gqa_backward(dout, cache, nh, nkv, hd)
print(f'\nBackward shapes: dq={dq.shape}, dk={dk.shape}, dv={dv.shape}')
print('  (dk and dv have n_kv_heads, not n_heads — gradients are summed across the group)')

## 3. Mixed-Precision (float16 forward)

When `mixed_precision=True`, the model casts master weights to float16
for the forward pass. Master weights remain float32 and gradients are
accumulated in float32.

Note: On CPU, float16 is actually slower (no SIMD benefit). This is
primarily for future GPU backends.

In [ ]:
cfg_mp = config_small()
cfg_mp.mixed_precision = True
model_mp = TinyGPTV3(cfg_mp)

tokens = np.array([1, 2, 3, 4, 5], dtype=np.int64)
with model_mp.mixed_precision(True):
    logits_mp, _ = model_mp.forward(tokens)
print(f'float16 forward logits shape: {logits_mp.shape}')
print(f'Logits dtype: {logits_mp.dtype}')
print(f'All finite: {np.all(np.isfinite(logits_mp))}')

# Compare with float32.
model_fp32 = TinyGPTV3(config_small())
model_fp32.token_emb = model_mp.token_emb.copy()
model_fp32.lm_head = model_mp.lm_head.copy()
for i in range(len(model_mp.layers)):
    for attr in ('W_q','W_k','W_v','W_o','b_q','b_k','b_v','b_o',
                 'ln1_gamma','ln1_beta','ln2_gamma','ln2_beta',
                 'W_fc1','W_fc2','b_fc1','b_fc2'):
        setattr(model_fp32.layers[i], attr, getattr(model_mp.layers[i], attr).copy())

logits_fp32, _ = model_fp32.forward(tokens)
rel_diff = np.max(np.abs(logits_mp - logits_fp32) / (np.abs(logits_fp32) + 1e-6))
print(f'\nMax relative difference (fp16 vs fp32): {rel_diff:.4e}')

## 4. Gradient Checkpointing

With `gradient_checkpointing=True`, the forward pass stores only the
layer inputs. During backward, each layer's intermediates are
recomputed. This trades ~49% backward slowdown for ~10× memory
reduction.

The gradients are **bit-for-bit identical** to the no-checkpointing path.

In [ ]:
cfg_ckpt = config_small()
cfg_ckpt.gradient_checkpointing = True
model_ckpt = TinyGPTV3(cfg_ckpt)

model_no_ckpt = TinyGPTV3(config_small())
# Copy weights.
model_ckpt.token_emb = model_no_ckpt.token_emb.copy()
model_ckpt.lm_head = model_no_ckpt.lm_head.copy()
for i in range(len(model_no_ckpt.layers)):
    for attr in ('W_q','W_k','W_v','W_o','b_q','b_k','b_v','b_o',
                 'ln1_gamma','ln1_beta','ln2_gamma','ln2_beta',
                 'W_fc1','W_fc2','b_fc1','b_fc2'):
        setattr(model_ckpt.layers[i], attr,
                getattr(model_no_ckpt.layers[i], attr).copy())

tokens = np.array([1, 2, 3, 4], dtype=np.int64)
l1, c1 = model_no_ckpt.forward_with_cache(tokens)
l2, c2 = model_ckpt.forward_with_cache(tokens)
print(f'Logits match: {np.allclose(l1, l2)}')

dlogits = np.random.randn(*l1.shape).astype(np.float32)
g1 = model_no_ckpt.backward(c1, dlogits.copy())
g2 = model_ckpt.backward(c2, dlogits.copy())
max_diff = max(np.max(np.abs(g1[k] - g2[k])) for k in g1 if k in g2)
print(f'Max |grad_no_ckpt - grad_ckpt| = {max_diff:.2e}')
print(f'Gradients identical: {max_diff < 1e-10}')

## 5. 4M-Parameter Config

The `config_4m()` preset targets ~4.15M parameters: 10 layers,
hidden=192, 6 query heads, 2 KV heads, BPE vocab=512.

In [ ]:
cfg = config_4m()
print(f'Config: {cfg}')
print(f'Parameters: {cfg.params_count:,}')
print(f'  GQA group size: {cfg.n_groups}')
print(f'  RoPE: {cfg.use_rope}')
print(f'  head_dim: {cfg.head_dim}')
print(f'  mlp_dim: {cfg.mlp_dim}')

model = TinyGPTV3(cfg)
tokens = np.array([1, 2, 3, 4, 5], dtype=np.int64)
logits, hidden = model.forward(tokens)
print(f'\nForward pass:')
print(f'  Logits shape: {logits.shape}')
print(f'  Hidden states: {len(hidden)} layers, each {hidden[0].shape}')
print(f'  All finite: {np.all(np.isfinite(logits))}')

## 6. Gradient Check (Finite-Difference Verification)

The hand-written backward pass is verified against central finite
differences in float64. The relative error should be < 1e-4 for all
parameters.

In [ ]:
# Run the gradient check script.
import subprocess
result = subprocess.run(
    ['python3', 'tests/v3/grad_check_precise.py'],
    capture_output=True, text=True, cwd='..'
)
# Print the last 10 lines.
lines = result.stdout.strip().split('\n')
for line in lines[-10:]:
    print(line)

## 7. Spectral Analysis (RMT Diagnostics)

The spectral_analysis method computes per-layer covariance eigenvalues
and compares them against Marchenko-Pastur bounds. A signal eigenvalue
above the MP upper edge indicates the BBP phase transition (cognitive
mode detection).

In [ ]:
model = TinyGPTV3(config_small())
tokens = np.arange(1, 33, dtype=np.int64)  # 32 tokens
_, hidden = model.forward(tokens)
spec = model.spectral_analysis(hidden)

print(f'Layers analyzed: {len(spec["layers"])}')
print(f'\n{"Layer":>5} {"λ_max":>10} {"λ_MP+":>10} {"q":>6} {"signal":>7} {"gap":>10}')
print('-' * 55)
for s in spec['layers']:
    sig = 'YES' if s['signal_detected'] else 'no'
    print(f'{s["layer"]:>5} {s["lambda_max"]:>10.4f} {s["mp_upper"]:>10.4f} '
          f'{s["q"]:>6.3f} {sig:>7} {s["spectral_gap"]:>10.4f}')

## Summary

TinyGPT v3 modernizes the v2 architecture with four production-grade
techniques:

| Feature | Benefit | ADR |
|---------|---------|-----|
| RoPE | Length generalization, saves pos_emb params | ADR-009 |
| GQA | Fewer KV params, smaller KV cache | ADR-010 |
| Mixed-precision | 2× speedup on GPU (future) | ADR-011 |
| Gradient checkpointing | 10× memory reduction | ADR-012 |

All gradients are verified by finite-difference check. The 4M-parameter
config is ready for training on the expanded corpus.